In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [4]:
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_decision_forests as tfdf

# =====================
# データ読み込み
# =====================
train_df = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
test_df  = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')

# =====================
# Name 正規化
# =====================
def normalize_name(x):
    return " ".join([v.strip(",()[].\"'") for v in x.split(" ")])

# =====================
# 前処理
# =====================
def preprocess(df):
    df = df.copy()

    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    df['Embarked'] = df['Embarked'].fillna('S')

    df['Name'] = df['Name'].fillna('UNKNOWN')
    df['Name'] = df['Name'].apply(normalize_name)

    df['Ticket_number'] = df['Ticket'].apply(lambda x: x.split()[-1])
    df['Ticket_item'] = df['Ticket'].apply(
        lambda x: 'NONE' if len(x.split()) == 1 else '_'.join(x.split()[:-1])
    )

    df['Ticket_number'] = pd.to_numeric(df['Ticket_number'], errors='coerce')
    df['Ticket_number'] = df['Ticket_number'].fillna(-1)

    df['Cabin'] = df['Cabin'].fillna('UNKNOWN')

    df['Sex'] = df['Sex'].fillna('UNKNOWN')
    df['Embarked'] = df['Embarked'].fillna('UNKNOWN')
    df['Ticket_item'] = df['Ticket_item'].fillna('UNKNOWN')

    return df

train_df = preprocess(train_df)
test_df = preprocess(test_df)

# =====================
# 使用特徴量
# =====================
features = [
    'Pclass',
    'Sex',
    'Age',
    'SibSp',
    'Parch',
    'Fare',
    'Embarked',
    'Cabin',
    'Ticket_number',
    'Ticket_item',
    'Name'
]

label = 'Survived'

train_data = train_df[features + [label]]
test_data = test_df[features]

# =====================
# TensorFlow Dataset化
# =====================
train_ds_base = tfdf.keras.pd_dataframe_to_tf_dataset(
    train_data,
    label=label
)

test_ds_base = tfdf.keras.pd_dataframe_to_tf_dataset(
    test_data
)

# =====================
# Nameをトークン化
# =====================
def tokenize_names(features, labels=None):
    features = dict(features)
    features["Name"] = tf.strings.split(features["Name"])

    if labels is None:
        return features
    return features, labels

train_ds = train_ds_base.map(tokenize_names)
test_ds = test_ds_base.map(tokenize_names)

# =====================
# seedセット定義
# =====================
seed_sets = {
    "seed0_20": range(0, 20),
    "seed0_15": range(0, 15),
    "seed0_12": range(0, 12),
    "seed0_10": range(0, 10),
    "seed2_20": range(2, 20),
    "seed3_20": range(3, 20),
}

# =====================
# threshold定義
# =====================
thresholds = [
    0.480, 0.482, 0.484, 0.486, 0.488,
    0.490, 0.491, 0.492, 0.493, 0.494,
    0.495, 0.496, 0.498, 0.500, 0.502
]

# =====================
# seedセットごとに学習・予測・submission作成
# =====================
for set_name, seeds in seed_sets.items():
    print(f"\n===== {set_name} 開始 =====")

    preds = []

    for seed in seeds:
        model = tfdf.keras.GradientBoostedTreesModel(
            random_seed=seed,
            honest=True,
            verbose=0
        )

        model.fit(train_ds)

        pred = model.predict(test_ds, verbose=0).flatten()
        preds.append(pred)

        print(f"seed {seed} 完了")

    final_pred = np.mean(preds, axis=0)

    for th in thresholds:
        submission = pd.DataFrame({
            'PassengerId': test_df['PassengerId'],
            'Survived': (final_pred > th).astype(int)
        })

        filename = f'tfdf_honest_{set_name}_th{th:.3f}.csv'
        submission.to_csv(filename, index=False)

        print(f'{filename} 作成完了')




===== seed0_20 開始 =====


W0000 00:00:1778500342.102819      57 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500342.102869      57 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500342.102873      57 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1778500342.416972      57 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1778500342.417038      57 kernel.cc:783] Collect training examples
I0000 00:00:1778500342.417055      57 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false


seed 0 完了


W0000 00:00:1778500343.334494      57 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500343.334530      57 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500343.334534      57 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1778500343.622634      57 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1778500343.622674      57 kernel.cc:783] Collect training examples
I0000 00:00:1778500343.622686      57 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false


seed 1 完了


...

seed 17 完了


W0000 00:00:1778500449.531511      57 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500449.531546      57 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500449.531551      57 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1778500449.823797      57 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1778500449.823832      57 kernel.cc:783] Collect training examples
I0000 00:00:1778500449.823843      57 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false


seed 18 完了


W0000 00:00:1778500450.470416      57 gradient_boosted_trees.cc:1873] "goss_alpha" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500450.470452      57 gradient_boosted_trees.cc:1883] "goss_beta" set but "sampling_method" not equal to "GOSS".
W0000 00:00:1778500450.470457      57 gradient_boosted_trees.cc:1897] "selective_gradient_boosting_ratio" set but "sampling_method" not equal to "SELGB".
I0000 00:00:1778500450.777009      57 kernel.cc:782] Start Yggdrasil model training
I0000 00:00:1778500450.777047      57 kernel.cc:783] Collect training examples
I0000 00:00:1778500450.777060      57 kernel.cc:795] Dataspec guide:
column_guides {
  column_name_pattern: "^__LABEL$"
  type: CATEGORICAL
  categorial {
    min_vocab_frequency: 0
    max_vocab_count: -1
  }
}
default_column_guide {
  categorial {
    max_vocab_count: 2000
  }
  discretized_numerical {
    maximum_num_bins: 255
  }
}
ignore_columns_without_guides: false
detect_numerical_as_discretized_numerical: false


seed 19 完了
tfdf_honest_seed3_20_th0.480.csv 作成完了
tfdf_honest_seed3_20_th0.482.csv 作成完了
tfdf_honest_seed3_20_th0.484.csv 作成完了
tfdf_honest_seed3_20_th0.486.csv 作成完了
tfdf_honest_seed3_20_th0.488.csv 作成完了
tfdf_honest_seed3_20_th0.490.csv 作成完了
tfdf_honest_seed3_20_th0.491.csv 作成完了
tfdf_honest_seed3_20_th0.492.csv 作成完了
tfdf_honest_seed3_20_th0.493.csv 作成完了
tfdf_honest_seed3_20_th0.494.csv 作成完了
tfdf_honest_seed3_20_th0.495.csv 作成完了
tfdf_honest_seed3_20_th0.496.csv 作成完了
tfdf_honest_seed3_20_th0.498.csv 作成完了
tfdf_honest_seed3_20_th0.500.csv 作成完了
tfdf_honest_seed3_20_th0.502.csv 作成完了
